## Hyperparameter Tuning Results

### Progression: Dummy → Tuned XGBoost

**Results:**

| Model | Precision | Recall | F1 | ROC-AUC | PR-AUC |
|------|----------:|-------:|---:|--------:|-------:|
| Dummy Classifier | 0.00 | 0.00 | 0.00 | 0.50 | 0.33 |
| Logistic Regression | 0.54 | 0.81 | 0.65 | 0.80 | 0.620 |
| XGBoost (`scale_pos_weight`, default) | 0.52 | 0.76 | 0.61 | 0.78 | 0.578 |
| XGBoost (SMOTE, default) | 0.52 | 0.77 | 0.62 | 0.77 | 0.565 |
| **XGBoost (Tuned)** | **0.51** | **0.85** | **0.64** | **0.80** | **0.604** |

Best cross-validated PR-AUC during search: 0.6118 (close to the 0.604 test-set PR-AUC, suggesting the tuning generalized well rather than overfitting to the CV folds).

Best hyperparameters found:
- subsample: 0.9
- n_estimators: 200
- min_child_weight: 1
- max_depth: 4
- learning_rate: 0.01
- gamma: 0.1
- colsample_bytree: 1.0

### Incident: scikit-learn API mismatch during tuning

The first run of `RandomizedSearchCV` produced `nan` for every single one of the 250 fold/parameter combinations tested. The traceback showed  `TypeError: got an unexpected keyword argument 'needs_proba'` — this argument to `make_scorer()` was deprecated in recent scikit-learn versions (1.7.2 here) in favor of `response_method='predict_proba'`. Because the scorer failed silently inside cross-validation rather than crashing the whole script, the search completed and printed a "best" set of parameters despite every score being invalid — a good reminder to check for `nan` in search results rather than assuming a successful-looking run produced real, usable results.

### Did tuning fix the "XGBoost underperforms Logistic Regression" problem?

Partially. Tuning improved XGBoost's PR-AUC from 0.578 (untuned scale_pos_weight) to 0.604 — a real, meaningful gain — but it still falls slightly short of Logistic Regression's 0.62. This confirms Day 3's hypothesis: part of the gap was due to untuned hyperparameters, but not all of it. The remaining gap is likely because the underlying churn signal in this dataset is fairly linear (features like Frequency and Monetary having a fairly direct relationship with churn risk), which favors a linear model, combined with a relatively small dataset (~3,100 training rows) that limits how much a tree ensemble's flexibility can pay off.

### Precision/recall tradeoff shift

The tuned XGBoost model shows a different tradeoff than Logistic Regression: higher recall (0.85 vs 0.81) but lower precision (0.51 vs 0.54). It catches more actual churners, at the cost of more false alarms (211 false positives vs 181 for Logistic Regression). Neither is objectively better without a business cost context — this is exactly what Day 5's threshold and CLV analysis will formalize.

### Model selection decision

Despite Logistic Regression's narrow edge on raw PR-AUC, XGBoost was selected going forward for two reasons: 
1. Week 3's SHAP explainability step works natively and efficiently with tree-based models (`TreeExplainer`), which is the standard approach for per-customer churn explanations in production churn systems.
2. The performance gap (0.604 vs 0.62) is small enough that the explainability and downstream flexibility benefits outweigh the marginal PR-AUC difference.

Logistic Regression remains documented as the baseline comparison rather than being discarded from the project narrative — it's a legitimate finding that a simpler model performed comparably.